# NB-05 | Benchmark: Baseline 5 — LLM Prompt Expansion + Props LoRA

Оценка **BL3 + LLM auto-expansion** для Props/Effects.  
KID считается относительно `reference/props`.  
CLIP считается по **расширенному** промпту из `expanded_prompts.json`.

In [1]:
import os, json, struct
from pathlib import Path
import torch, numpy as np, mlflow
from PIL import Image
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
import open_clip
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [2]:
BASELINE_ID   = 'baseline_5'
GENERATED_DIR = Path('generated/baseline_5')
REFERENCE_DIR = Path('reference/props')  # <-- props!

EXPANDED_PROMPTS_JSON = Path('generated/baseline_5/expanded_prompts.json')

MLFLOW_URI      = 'http://127.0.0.1:5000'
EXPERIMENT_NAME = 'anima_baseline_benchmarks'

MODEL_PARAMS = {
    'model.name':       'anima_base',
    'model.type':       'props_lora_llm',
    'model.props_lora': 'spll_icn_props_v1.safetensors',
    'model.trigger':    '@spll_icn',
    'model.lora_stack': 'turbo+props',
}

INFER_PARAMS = {
    'infer.baseline_id':   BASELINE_ID,
    'infer.sampler':       'euler_ancestral',
    'infer.scheduler':     'normal',
    'infer.steps':         12,
    'infer.cfg':           2.0,
    'infer.turbo_strength': 1.0,
    'infer.style_strength': 0.85,
    'infer.llm_expansion': True,
    'infer.llm_model':     'mistral',
    'infer.n_images':      100,
}

print('Config loaded ✓')

Config loaded ✓


In [3]:
if EXPANDED_PROMPTS_JSON.exists():
    with open(EXPANDED_PROMPTS_JSON) as f:
        expanded_data = json.load(f)
    PROMPTS_FOR_CLIP = [d['expanded_prompt'] for d in expanded_data]
    print(f'Expanded prompts loaded: {len(PROMPTS_FOR_CLIP)}')
else:
    print('WARNING: expanded_prompts.json not found — using raw props prompts')
    BASE = [
        'glowing magic orb, blue energy, dark background, no humans',
        'fire spell icon, orange glow, dark background, no humans',
        'ice crystal, cyan light, dark background, no humans',
        'lightning rune, yellow glow, dark background, no humans',
        'healing potion, green glow, dark background, no humans',
        'dark magic circle, purple energy, dark background, no humans',
        'glowing sword, blue outline, dark background, no humans',
        'fire aura, red orange, simple background, no humans',
        'water elemental, blue mist, dark background, no humans',
        'magic crystal, pink glow, dark background, no humans',
    ]
    PROMPTS_FOR_CLIP = [BASE[i % 10] for i in range(100)]

Expanded prompts loaded: 100


In [4]:
def load_t(folder, size=(299,299)):
    return torch.stack([torch.tensor(np.array(Image.open(p).convert('RGB').resize(size))).permute(2,0,1) for p in sorted(folder.glob('*.png'))])

def load_pil(folder, limit=None):
    files=sorted(folder.glob('*.png'))
    if limit: files=files[:limit]
    return [Image.open(p).convert('RGB') for p in files]

def make_grid(imgs, n=8, thumb=128):
    imgs=[i.resize((thumb,thumb)) for i in imgs[:n]]
    c=Image.new('RGB',(thumb*len(imgs),thumb),(20,20,20))
    for i,im in enumerate(imgs): c.paste(im,(thumb*i,0))
    return c

def compute_kid(gen, ref, subset=50):
    k=KernelInceptionDistance(subset_size=subset,normalize=True).to(DEVICE)
    k.update(load_t(ref).to(DEVICE),real=True)
    k.update(load_t(gen).to(DEVICE),real=False)
    m,s=k.compute(); return float(m.cpu()),float(s.cpu())

def compute_clip(gen, prompts):
    model,_,prep=open_clip.create_model_and_transforms('ViT-B-32',pretrained='openai')
    model=model.to(DEVICE).eval()
    tok=open_clip.get_tokenizer('ViT-B-32')
    sc=[]
    for i,p in enumerate(tqdm(sorted(gen.glob('*.png')),'CLIP')):
        if i>=len(prompts): break
        img=prep(Image.open(p).convert('RGB')).unsqueeze(0).to(DEVICE)
        txt=tok([prompts[i]]).to(DEVICE)
        with torch.no_grad():
            a=model.encode_image(img);a/=a.norm(dim=-1,keepdim=True)
            b=model.encode_text(txt);b/=b.norm(dim=-1,keepdim=True)
            sc.append(float((a*b).sum().cpu()))
    return float(np.mean(sc))

def compute_lpips(gen, n_pairs=200):
    lp=LearnedPerceptualImagePatchSimilarity(net_type='vgg',normalize=True).to(DEVICE)
    files=sorted(gen.glob('*.png'))
    idx=np.random.default_rng(42).integers(0,len(files),size=(n_pairs,2))
    sc=[]
    def tt(p): return torch.tensor(np.array(Image.open(p).convert('RGB').resize((256,256))).astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(DEVICE)
    for a,b in tqdm(idx,'LPIPS'):
        if a==b: continue
        with torch.no_grad(): sc.append(float(lp(tt(files[a]),tt(files[b])).cpu()))
    return float(np.mean(sc))

print('Helpers ready ✓')

Helpers ready ✓


In [5]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f'eval_{BASELINE_ID}_propsLoRA_llmExpand'

with mlflow.start_run(run_name=run_name) as run:
    print(f'Run ID: {run.info.run_id}')

    mlflow.set_tags({
        'evaluation_type': 'benchmark',
        'training_logged_retroactively': 'true',
        'family':          'props',
        'baseline_id':     BASELINE_ID,
        'llm_expansion':   'true',
        'environment':     'local_rtx4060ti_16gb',
    })

    mlflow.log_params(MODEL_PARAMS)
    mlflow.log_params(INFER_PARAMS)
    mlflow.log_param('data.reference_dir', str(REFERENCE_DIR))
    mlflow.log_param('data.clip_evaluated_on', 'expanded_prompts')

    if EXPANDED_PROMPTS_JSON.exists():
        mlflow.log_artifact(str(EXPANDED_PROMPTS_JSON), artifact_path='prompts')

    kid_mean, kid_std = compute_kid(GENERATED_DIR, REFERENCE_DIR)
    clip_score        = compute_clip(GENERATED_DIR, PROMPTS_FOR_CLIP)
    lpips_div         = compute_lpips(GENERATED_DIR)

    if lpips_div < 0.10:
        mlflow.set_tag('warning', 'possible_mode_collapse_lpips_below_0.10')
        print('⚠ WARNING: possible mode collapse!')

    mlflow.log_metrics({
        'eval.kid_mean':        kid_mean,
        'eval.kid_std':         kid_std,
        'eval.clip_score':      clip_score,
        'eval.lpips_diversity': lpips_div,
        'eval.n_generated':     float(len(list(GENERATED_DIR.glob('*.png')))),
    })

    grid = make_grid(load_pil(GENERATED_DIR, limit=8))
    Path('tmp').mkdir(parents=True, exist_ok=True)
    gp = f'tmp/{BASELINE_ID}_grid.png'; grid.save(gp)
    mlflow.log_artifact(gp, artifact_path='samples')

    print(f'\n=== BL5 Results ===')
    print(f'KID:   {kid_mean:.6f} ± {kid_std:.6f}')
    print(f'CLIP:  {clip_score:.4f}  (measured on expanded prompts)')
    print(f'LPIPS: {lpips_div:.4f}')
    print('✓ Logged to MLflow')

Run ID: 677787ada13a443d820780245679e18d


e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Metric `Kernel Inception Distance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
LPIPS: 100%|██████████| 200/200 [01:03<00:00,  3.15it/s]


=== BL5 Results ===
KID:   0.036467 ± 0.004849
CLIP:  0.2710  (measured on expanded prompts)
LPIPS: 0.6659
✓ Logged to MLflow
🏃 View run eval_baseline_5_propsLoRA_llmExpand at: http://127.0.0.1:5000/#/experiments/1/runs/677787ada13a443d820780245679e18d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
